# Healthcare Claims — SQL Analytics

Running the SQL queries from `sql/medicaid_analytics.sql` using SQLite in-memory,
so you can see actual output without needing a database server.

Same queries work on Databricks SQL, Snowflake, or BigQuery — just swap the connection.

In [ ]:
import sqlite3
import pandas as pd

pd.set_option('display.float_format', '{:,.2f}'.format)

conn = sqlite3.connect(':memory:')
df   = pd.read_csv('../data/sample_claims.csv')
df.to_sql('claims', conn, index=False, if_exists='replace')
print(f'Loaded {len(df)} rows into in-memory SQLite')

def q(sql):
    return pd.read_sql(sql, conn)

## Q1 — Baseline KPIs

In [ ]:
q("""
SELECT
    COUNT(*)                                                          AS total_claims,
    COUNT(CASE WHEN claim_status='PAID'   THEN 1 END)                 AS paid_claims,
    COUNT(CASE WHEN claim_status='DENIED' THEN 1 END)                 AS denied_claims,
    ROUND(100.0 * COUNT(CASE WHEN claim_status='DENIED' THEN 1 END)
                / COUNT(*), 2)                                        AS denial_rate_pct,
    ROUND(SUM(billed_amount), 2)                                      AS total_billed,
    ROUND(SUM(paid_amount), 2)                                        AS total_paid,
    ROUND(100.0 * SUM(paid_amount) / SUM(billed_amount), 2)           AS avg_payment_rate_pct
FROM claims
""")

## Q2 — Monthly Spend Trend

In [ ]:
q("""
SELECT
    STRFTIME('%Y-%m', service_date) AS service_month,
    claim_type,
    COUNT(*)                        AS claim_count,
    ROUND(SUM(paid_amount), 2)      AS total_paid,
    ROUND(AVG(paid_amount), 2)      AS avg_paid_per_claim
FROM claims
WHERE claim_status = 'PAID'
GROUP BY 1, 2
ORDER BY 1, 2
""")

## Q3 — Spend by State and Plan (PMPM proxy)

In [ ]:
q("""
SELECT
    state_code,
    plan_id,
    COUNT(DISTINCT member_id)                                          AS unique_members,
    COUNT(*)                                                           AS total_claims,
    ROUND(SUM(paid_amount), 2)                                         AS total_paid,
    ROUND(SUM(paid_amount) / COUNT(DISTINCT member_id), 2)             AS pmpm_proxy
FROM claims
WHERE claim_status = 'PAID'
GROUP BY 1, 2
ORDER BY total_paid DESC
""")

## Q4 — High-Cost Members

In [ ]:
q("""
SELECT
    member_id,
    state_code,
    COUNT(*)                                           AS total_claims,
    COUNT(CASE WHEN claim_type='IP' THEN 1 END)        AS inpatient_visits,
    ROUND(SUM(paid_amount), 2)                         AS total_paid,
    ROUND(AVG(paid_amount), 2)                         AS avg_per_claim
FROM claims
WHERE claim_status = 'PAID'
GROUP BY 1, 2
HAVING SUM(paid_amount) > 5000
ORDER BY total_paid DESC
""")

## Q5 — Denial Rate by Provider Type

In [ ]:
q("""
SELECT
    provider_type,
    COUNT(*)                                                             AS total_claims,
    COUNT(CASE WHEN claim_status='DENIED' THEN 1 END)                    AS denied,
    ROUND(100.0 * COUNT(CASE WHEN claim_status='DENIED' THEN 1 END)
                / COUNT(*), 2)                                           AS denial_rate_pct
FROM claims
GROUP BY 1
ORDER BY denial_rate_pct DESC
""")

## Q6 — Data Quality: Denied Claims with Paid Amount

In [ ]:
q("""
SELECT claim_id, member_id, service_date, billed_amount, paid_amount, claim_status,
       'denied but paid_amount > 0' AS anomaly_flag
FROM claims
WHERE claim_status = 'DENIED' AND paid_amount > 0
""")

## Q7 — 30-Day Readmission Proxy

Members with two inpatient claims within 30 days. Not a perfect readmission measure (would need discharge dates) but a useful signal.

In [ ]:
q("""
SELECT
    a.member_id,
    a.claim_id                                                     AS first_admit,
    a.service_date                                                 AS first_date,
    b.claim_id                                                     AS readmit_claim,
    b.service_date                                                 AS readmit_date,
    CAST(JULIANDAY(b.service_date)-JULIANDAY(a.service_date) AS INT) AS days_between
FROM claims a
JOIN claims b
  ON  a.member_id = b.member_id AND a.claim_id != b.claim_id
  AND a.claim_type = 'IP' AND b.claim_type = 'IP'
  AND b.service_date > a.service_date
  AND JULIANDAY(b.service_date)-JULIANDAY(a.service_date) <= 30
WHERE a.claim_status='PAID' AND b.claim_status='PAID'
ORDER BY a.member_id
""")